In [1]:
!pip install -U langchain langgraph langchain-google-genai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 19.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.9
    Uninstalling langgraph-1.2.9:
      Successfully uninstalled langgraph-1.2.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.13
    Uninstalling langchain-1.3.13:
      Successfully uninstalled langchain-1.3.13


In [2]:
import os
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)
os.environ["GOOGLE_API_KEY"] = api_key
#import os
#from getpass import getpass

#os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API key: ")

In [3]:
"""
LangChain + LangGraph Vehicle Telemetry Agent System

Demonstrates:

1. Main orchestration agent
2. Static subagents
3. Dynamic subagents
4. LangGraph state management
5. Conditional routing
6. LangChain tools
7. LLM-based specialist diagnosis
8. Final report aggregation

Install:

    pip install -U langchain langgraph \
        langchain-google-genai pydantic

Environment:

    GOOGLE_API_KEY=<your Gemini API key>
"""

from __future__ import annotations

import json
import os
from datetime import datetime
from operator import add
from typing import Annotated, Any, Literal, TypedDict

from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

from pydantic import BaseModel, Field

In [4]:
# ============================================================
# 1. DOMAIN MODELS
# ============================================================

class TelemetryReading(BaseModel):
    vehicle_id: str
    timestamp: str

    speed_kmph: float
    engine_temperature_c: float
    battery_voltage_v: float
    oil_pressure_psi: float
    brake_pad_percent: float
    tire_pressure_psi: float
    fuel_percent: float


class Finding(BaseModel):
    agent_name: str
    agent_type: str
    component: str
    severity: Literal["NORMAL", "WARNING", "CRITICAL"]
    observation: str
    recommendation: str


class HealthAssessment(BaseModel):
    overall_status: Literal["NORMAL", "WARNING", "CRITICAL"]

    required_specialists: list[
        Literal[
            "battery",
            "thermal",
            "brake",
            "engine",
            "tire"
        ]
    ] = Field(default_factory=list)

    findings: list[Finding] = Field(default_factory=list)

In [6]:
# ============================================================
# 2. LANGGRAPH SHARED STATE
# ============================================================

class TelemetryState(TypedDict, total=False):
    """
    Shared state passed between LangGraph nodes.
    """

    reading: dict[str, Any]

    validation_passed: bool
    validation_messages: list[str]

    health_assessment: dict[str, Any]
    required_specialists: list[str]

    # add reducer allows multiple dynamic agents to append results.
    specialist_findings: Annotated[list[dict[str, Any]], add]

    execution_trace: Annotated[list[str], add]

    final_report: str


class SpecialistState(TypedDict):
    """
    State sent separately to each dynamically created specialist.
    """

    reading: dict[str, Any]
    specialist_type: str


In [ ]:
# ============================================================
# 3. LLM CONFIGURATION
# ============================================================

if not os.getenv("GOOGLE_API_KEY"):
    raise EnvironmentError(
        "GOOGLE_API_KEY is not configured. "
        "Set it before running the program."
    )


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


# ============================================================
# 4. LANGCHAIN TOOLS
# ============================================================

@tool
def validate_telemetry_values(
    speed_kmph: float,
    engine_temperature_c: float,
    battery_voltage_v: float,
    oil_pressure_psi: float,
    brake_pad_percent: float,
    tire_pressure_psi: float,
    fuel_percent: float
) -> str:
    """
    Validate whether vehicle telemetry values are within
    physically reasonable input ranges.
    """

    rules = {
        "speed_kmph": (speed_kmph, 0, 300),
        "engine_temperature_c": (
            engine_temperature_c,
            -40,
            180
        ),
        "battery_voltage_v": (
            battery_voltage_v,
            0,
            30
        ),
        "oil_pressure_psi": (
            oil_pressure_psi,
            0,
            150
        ),
        "brake_pad_percent": (
            brake_pad_percent,
            0,
            100
        ),
        "tire_pressure_psi": (
            tire_pressure_psi,
            0,
            100
        ),
        "fuel_percent": (
            fuel_percent,
            0,
            100
        ),
    }

    errors = []

    for field_name, values in rules.items():
        value, minimum, maximum = values

        if not minimum <= value <= maximum:
            errors.append(
                f"{field_name}={value} is outside "
                f"{minimum}–{maximum}"
            )

    if errors:
        return json.dumps(
            {
                "valid": False,
                "errors": errors
            }
        )

    return json.dumps(
        {
            "valid": True,
            "errors": []
        }
    )


@tool
def calculate_vehicle_faults(
    engine_temperature_c: float,
    battery_voltage_v: float,
    oil_pressure_psi: float,
    brake_pad_percent: float,
    tire_pressure_psi: float
) -> str:
    """
    Apply deterministic engineering thresholds and identify
    potential vehicle faults.
    """

    faults = []

    if engine_temperature_c >= 115:
        faults.append(
            {
                "specialist": "thermal",
                "component": "Engine cooling",
                "severity": "CRITICAL",
                "message": (
                    "Engine temperature is above the "
                    "critical threshold."
                )
            }
        )

    elif engine_temperature_c >= 105:
        faults.append(
            {
                "specialist": "thermal",
                "component": "Engine cooling",
                "severity": "WARNING",
                "message": "Engine temperature is elevated."
            }
        )

    if battery_voltage_v < 11.8:
        faults.append(
            {
                "specialist": "battery",
                "component": "Battery",
                "severity": "CRITICAL",
                "message": "Battery voltage is critically low."
            }
        )

    elif battery_voltage_v < 12.2:
        faults.append(
            {
                "specialist": "battery",
                "component": "Battery",
                "severity": "WARNING",
                "message": "Battery voltage is below normal."
            }
        )

    if oil_pressure_psi < 20:
        faults.append(
            {
                "specialist": "engine",
                "component": "Lubrication system",
                "severity": "CRITICAL",
                "message": "Engine oil pressure is dangerously low."
            }
        )

    if brake_pad_percent <= 10:
        faults.append(
            {
                "specialist": "brake",
                "component": "Braking system",
                "severity": "CRITICAL",
                "message": "Brake pads are critically worn."
            }
        )

    elif brake_pad_percent <= 20:
        faults.append(
            {
                "specialist": "brake",
                "component": "Braking system",
                "severity": "WARNING",
                "message": "Brake pads are approaching replacement."
            }
        )

    if tire_pressure_psi < 28:
        faults.append(
            {
                "specialist": "tire",
                "component": "Tires",
                "severity": "WARNING",
                "message": "Tire pressure is below normal."
            }
        )

    return json.dumps(
        {
            "faults": faults,
            "fault_count": len(faults)
        }
    )


# ============================================================
# 5. STATIC LANGCHAIN SUBAGENTS
# ============================================================

validation_subagent = create_agent(
    model=llm,
    tools=[validate_telemetry_values],
    system_prompt="""
You are the Vehicle Telemetry Validation Subagent.

You are a permanent static subagent.

Your responsibility is to:

1. Read the supplied vehicle telemetry.
2. Call validate_telemetry_values.
3. Determine whether the data is valid.
4. Do not diagnose vehicle faults.
5. Return a concise validation result.

Always use the validation tool.
"""
)


health_analysis_subagent = create_agent(
    model=llm,
    tools=[calculate_vehicle_faults],
    system_prompt="""
You are the Vehicle Health Analysis Subagent.

You are a permanent static subagent.

Your responsibility is to:

1. Read validated telemetry.
2. Call calculate_vehicle_faults.
3. Identify abnormal components.
4. Identify which specialist types are required.
5. Do not invent sensor readings.
6. Base your classification primarily on the tool output.

Possible specialist names:

- battery
- thermal
- brake
- engine
- tire
"""
)


report_subagent = create_agent(
    model=llm,
    tools=[],
    system_prompt="""
You are the Vehicle Maintenance Report Subagent.

You are a permanent static subagent.

Create a clear engineering report using only the supplied facts.

Your report must include:

1. Vehicle information
2. Overall status
3. Sensor readings
4. General health findings
5. Specialist diagnoses
6. Recommended actions
7. Agents executed
8. Immediate safety actions

Do not invent facts or sensor measurements.
"""
)


# ============================================================
# 6. STATIC LANGGRAPH NODES
# ============================================================

def validation_node(
    state: TelemetryState
) -> dict[str, Any]:
    """
    Static subagent node:
    validates every telemetry request.
    """

    reading = TelemetryReading.model_validate(
        state["reading"]
    )

    prompt = f"""
Validate this telemetry reading.

Telemetry:

{reading.model_dump_json(indent=2)}

Call the validation tool and clearly state whether the
input data is valid.
"""

    response = validation_subagent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    final_message = response["messages"][-1]
    message_text = extract_message_text(final_message)

    # Deterministic validation is repeated here so graph routing
    # never depends only on free-form LLM wording.
    validation_result = json.loads(
        validate_telemetry_values.invoke(
            {
                "speed_kmph": reading.speed_kmph,
                "engine_temperature_c":
                    reading.engine_temperature_c,
                "battery_voltage_v":
                    reading.battery_voltage_v,
                "oil_pressure_psi":
                    reading.oil_pressure_psi,
                "brake_pad_percent":
                    reading.brake_pad_percent,
                "tire_pressure_psi":
                    reading.tire_pressure_psi,
                "fuel_percent":
                    reading.fuel_percent,
            }
        )
    )

    return {
        "validation_passed": validation_result["valid"],
        "validation_messages": (
            validation_result["errors"]
            if validation_result["errors"]
            else [message_text]
        ),
        "execution_trace": [
            "Static Data Validation Subagent"
        ]
    }


def health_analysis_node(
    state: TelemetryState
) -> dict[str, Any]:
    """
    Static subagent node:
    performs general health classification.
    """

    reading = TelemetryReading.model_validate(
        state["reading"]
    )

    prompt = f"""
Analyse the following vehicle telemetry.

Telemetry:

{reading.model_dump_json(indent=2)}

Use the fault calculation tool.

Explain:

1. Which systems are abnormal
2. Their severity
3. Which specialists should be requested
"""

    response = health_analysis_subagent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    message_text = extract_message_text(
        response["messages"][-1]
    )

    # Tool calculation is used as structured control data.
    deterministic_result = json.loads(
        calculate_vehicle_faults.invoke(
            {
                "engine_temperature_c":
                    reading.engine_temperature_c,
                "battery_voltage_v":
                    reading.battery_voltage_v,
                "oil_pressure_psi":
                    reading.oil_pressure_psi,
                "brake_pad_percent":
                    reading.brake_pad_percent,
                "tire_pressure_psi":
                    reading.tire_pressure_psi,
            }
        )
    )

    faults = deterministic_result["faults"]

    required_specialists = list(
        dict.fromkeys(
            fault["specialist"]
            for fault in faults
        )
    )

    severity_order = {
        "NORMAL": 0,
        "WARNING": 1,
        "CRITICAL": 2
    }

    overall_status = "NORMAL"

    for fault in faults:
        if (
            severity_order[fault["severity"]]
            > severity_order[overall_status]
        ):
            overall_status = fault["severity"]

    assessment = {
        "overall_status": overall_status,
        "faults": faults,
        "llm_analysis": message_text
    }

    return {
        "health_assessment": assessment,
        "required_specialists": required_specialists,
        "execution_trace": [
            "Static Vehicle Health Subagent"
        ]
    }


def invalid_report_node(
    state: TelemetryState
) -> dict[str, Any]:
    """
    Produces a report when input telemetry is invalid.
    """

    report = [
        "=" * 70,
        "INVALID TELEMETRY REPORT",
        "=" * 70,
        "",
        "The telemetry reading failed validation.",
        "",
        "Validation errors:"
    ]

    for message in state.get(
        "validation_messages",
        []
    ):
        report.append(f"- {message}")

    report.extend(
        [
            "",
            "Vehicle diagnosis was not executed because the "
            "input data may be unreliable.",
            "=" * 70
        ]
    )

    return {
        "final_report": "\n".join(report),
        "execution_trace": [
            "Invalid Input Report Node"
        ]
    }


# ============================================================
# 7. DYNAMIC SUBAGENT FACTORY
# ============================================================

DYNAMIC_SPECIALIST_PROMPTS = {
    "battery": """
You are a dynamically created Automotive Battery Specialist.

Diagnose battery and charging-system problems.

Consider:

- battery voltage
- state of charge
- alternator problems
- terminal or wiring faults
- urgency of maintenance

Use only the supplied telemetry.
""",

    "thermal": """
You are a dynamically created Automotive Thermal Specialist.

Diagnose overheating and cooling-system problems.

Consider:

- coolant level
- radiator fan
- water pump
- thermostat
- engine load
- shutdown risk

Use only the supplied telemetry.
""",

    "brake": """
You are a dynamically created Automotive Brake Specialist.

Diagnose brake-wear and safety problems.

Consider:

- brake-pad percentage
- urgency of replacement
- vehicle-operating restrictions
- inspection recommendations

Use only the supplied telemetry.
""",

    "engine": """
You are a dynamically created Engine Lubrication Specialist.

Diagnose low oil-pressure problems.

Consider:

- oil level
- leakage
- pressure sensor
- oil pump
- filter restriction
- internal engine wear
- immediate shutdown risk

Use only the supplied telemetry.
""",

    "tire": """
You are a dynamically created Tire Safety Specialist.

Diagnose tire-pressure problems.

Consider:

- underinflation
- possible puncture
- stability
- fuel efficiency
- tire wear
- recommended inspection

Use only the supplied telemetry.
"""
}


def create_dynamic_specialist(
    specialist_type: str
):
    """
    Creates a LangChain agent at runtime.

    This object is not created during application startup.
    """

    system_prompt = DYNAMIC_SPECIALIST_PROMPTS.get(
        specialist_type
    )

    if system_prompt is None:
        raise ValueError(
            f"Unsupported specialist type: {specialist_type}"
        )

    print(
        f"    [DYNAMIC CREATE] "
        f"{specialist_type.title()} Specialist"
    )

    agent = create_agent(
        model=llm,
        tools=[],
        system_prompt=system_prompt
    )

    return agent


# ============================================================
# 8. DYNAMIC SPECIALIST NODE
# ============================================================

def dynamic_specialist_node(
    state: SpecialistState
) -> dict[str, Any]:
    """
    This node receives one specialist request.

    A new LangChain subagent is instantiated at runtime,
    invoked and then released.
    """

    specialist_type = state["specialist_type"]
    reading = TelemetryReading.model_validate(
        state["reading"]
    )

    # Dynamic creation happens here.
    dynamic_agent = create_dynamic_specialist(
        specialist_type
    )

    prompt = f"""
Perform a specialist diagnosis for the following vehicle.

Specialist type:
{specialist_type}

Telemetry:

{reading.model_dump_json(indent=2)}

Return:

1. Component assessed
2. Severity
3. Probable causes
4. Immediate action
5. Maintenance recommendation
"""

    response = dynamic_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    diagnosis_text = extract_message_text(
        response["messages"][-1]
    )

    finding = {
        "agent_name": (
            f"Dynamic {specialist_type.title()} Specialist"
        ),
        "agent_type": "Dynamic Subagent",
        "specialist_type": specialist_type,
        "diagnosis": diagnosis_text
    }

    print(
        f"    [DYNAMIC RELEASE] "
        f"{specialist_type.title()} Specialist"
    )

    # The reference is intentionally not stored.
    del dynamic_agent

    return {
        "specialist_findings": [finding],
        "execution_trace": [
            f"Dynamic {specialist_type.title()} Specialist"
        ]
    }


# ============================================================
# 9. CONDITIONAL ROUTING
# ============================================================

def route_after_validation(
    state: TelemetryState
) -> Literal[
    "health_analysis",
    "invalid_report"
]:
    """
    Branch based on telemetry validity.
    """

    if state.get("validation_passed", False):
        return "health_analysis"

    return "invalid_report"


def dispatch_dynamic_specialists(
    state: TelemetryState
):
    """
    Dynamically dispatch one specialist node per detected fault.

    LangGraph Send allows multiple runtime tasks to be created.
    """

    specialists = state.get(
        "required_specialists",
        []
    )

    if not specialists:
        return "report"

    return [
        Send(
            "dynamic_specialist",
            {
                "reading": state["reading"],
                "specialist_type": specialist
            }
        )
        for specialist in specialists
    ]


# ============================================================
# 10. FINAL REPORT NODE
# ============================================================

def report_node(
    state: TelemetryState
) -> dict[str, Any]:
    """
    Static LangChain report-generation subagent.
    """

    reading = state["reading"]
    assessment = state.get(
        "health_assessment",
        {}
    )
    specialist_findings = state.get(
        "specialist_findings",
        []
    )
    trace = state.get(
        "execution_trace",
        []
    )

    prompt = f"""
Create the final vehicle telemetry health report.

Vehicle telemetry:

{json.dumps(reading, indent=2)}

General health assessment:

{json.dumps(assessment, indent=2)}

Dynamic specialist diagnoses:

{json.dumps(specialist_findings, indent=2)}

Executed agents:

{json.dumps(trace, indent=2)}

Clearly distinguish:

- Static subagents
- Dynamic subagents
- Overall vehicle status
- Immediate safety actions
"""

    response = report_subagent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    final_report = extract_message_text(
        response["messages"][-1]
    )

    return {
        "final_report": final_report,
        "execution_trace": [
            "Static Report Generation Subagent"
        ]
    }


# ============================================================
# 11. HELPER FUNCTION
# ============================================================

def extract_message_text(message: AIMessage) -> str:
    """
    Handles normal text and provider-specific content blocks.
    """

    content = message.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts = []

        for part in content:
            if isinstance(part, str):
                text_parts.append(part)

            elif isinstance(part, dict):
                text = part.get("text")

                if text:
                    text_parts.append(text)

        return "\n".join(text_parts)

    return str(content)


# ============================================================
# 12. BUILD THE LANGGRAPH WORKFLOW
# ============================================================

def build_telemetry_graph():
    """
    Build and compile the main orchestration graph.
    """

    builder = StateGraph(TelemetryState)

    # Static workflow nodes
    builder.add_node(
        "validation",
        validation_node
    )

    builder.add_node(
        "health_analysis",
        health_analysis_node
    )

    builder.add_node(
        "invalid_report",
        invalid_report_node
    )

    builder.add_node(
        "report",
        report_node
    )

    # Dynamic worker node
    builder.add_node(
        "dynamic_specialist",
        dynamic_specialist_node
    )

    # Main workflow
    builder.add_edge(
        START,
        "validation"
    )

    builder.add_conditional_edges(
        "validation",
        route_after_validation,
        {
            "health_analysis": "health_analysis",
            "invalid_report": "invalid_report"
        }
    )

    # Dynamic fan-out
    builder.add_conditional_edges(
        "health_analysis",
        dispatch_dynamic_specialists,
        ["dynamic_specialist", "report"]
    )

    # Every dynamically dispatched specialist returns here.
    builder.add_edge(
        "dynamic_specialist",
        "report"
    )

    builder.add_edge(
        "invalid_report",
        END
    )

    builder.add_edge(
        "report",
        END
    )

    return builder.compile()


telemetry_graph = build_telemetry_graph()


# ============================================================
# 13. TEST DATA
# ============================================================

def healthy_vehicle() -> TelemetryReading:
    return TelemetryReading(
        vehicle_id="TRUCK-101",
        timestamp=datetime.now().isoformat(),
        speed_kmph=72,
        engine_temperature_c=92,
        battery_voltage_v=12.8,
        oil_pressure_psi=42,
        brake_pad_percent=68,
        tire_pressure_psi=33,
        fuel_percent=61
    )


def faulty_vehicle() -> TelemetryReading:
    return TelemetryReading(
        vehicle_id="TRUCK-202",
        timestamp=datetime.now().isoformat(),
        speed_kmph=76,
        engine_temperature_c=119,
        battery_voltage_v=11.4,
        oil_pressure_psi=16,
        brake_pad_percent=8,
        tire_pressure_psi=25,
        fuel_percent=32
    )


# ============================================================
# 14. RUN THE APPLICATION
# ============================================================

def run_demo(
    reading: TelemetryReading
) -> None:

    print("\n" + "=" * 75)
    print(
        f"PROCESSING VEHICLE: {reading.vehicle_id}"
    )
    print("=" * 75)

    initial_state: TelemetryState = {
        "reading": reading.model_dump(),
        "validation_passed": False,
        "validation_messages": [],
        "health_assessment": {},
        "required_specialists": [],
        "specialist_findings": [],
        "execution_trace": [],
        "final_report": ""
    }

    result = telemetry_graph.invoke(initial_state)

    print("\nFINAL REPORT")
    print("=" * 75)
    print(result["final_report"])

    print("\nAGENT EXECUTION TRACE")
    print("=" * 75)

    for number, agent_name in enumerate(
        result.get("execution_trace", []),
        start=1
    ):
        print(f"{number}. {agent_name}")


def main() -> None:

    print("\n\nDEMO 1: HEALTHY VEHICLE")
    run_demo(healthy_vehicle())

    print("\n\nDEMO 2: VEHICLE WITH FAULTS")
    run_demo(faulty_vehicle())


if __name__ == "__main__":
    main()